In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from scripts.assetsRoster import carteira_AC, carteira_HB, carteira_EXC, carteira_LC, others, ROSTER
# Import the refactored TrendAnalyzer
from soros_system.main import TrendAnalyzer
from soros_system.analysis.markov_vol_model import MarkovVolModel

# Set paths
data_path='/Users/valter.rebelo/MissionControl/data/micro/candleData/'
btc_data_path='/Users/valter.rebelo/MissionControl/data/micro/candleData/bitcoin_candles.csv'
ssr_data_path='/Users/valter.rebelo/MissionControl/data/onchainData/BTC_SSR.csv'
market_data_path="/Users/valter.rebelo/MissionControl/data/micro/assetData/"
markov_vol_model = MarkovVolModel.load_model_by_timestamp('20250325_144631')

# Initialize the analyzer with a smaller set of assets for initial testing
test_assets = ['bitcoin', 'ethereum', 'solana']
print(f"Testing with assets: {test_assets}")

analyzer = TrendAnalyzer(
    asset_ids=test_assets,
    data_path=data_path,
    btc_data_path=btc_data_path,
    lookback_days='all',
    ssr_data_path=ssr_data_path,        
    markov_analyzer=markov_vol_model,
    market_data_path=market_data_path  
)

analyzer.analyze_multiple_assets()

Detected MarkovVolModel passed as markov_analyzer - using it for volatility analysis


Testing with assets: ['bitcoin', 'ethereum', 'solana']


Analyzing assets:   0%|          | 0/3 [00:00<?, ?it/s]

Classifying bitcoin:   0%|          | 0/1 [00:00<?, ?it/s]

Creating trend data for bitcoin:   0%|          | 0/2 [00:00<?, ?it/s]

...

Classifying solana:   0%|          | 0/1 [00:00<?, ?it/s]

Creating trend data for solana:   0%|          | 0/2 [00:00<?, ?it/s]

Computing metrics:   0%|          | 0/3 [00:00<?, ?it/s]

Column overall_trend_BTC not found in data for bitcoin. Skipping BTC trend metrics calculation.


['bitcoin', 'ethereum', 'solana']

In [32]:
# Import the signal registry
from soros_system.signals.signal_registry import get_all_signals, get_signal_info

# List all registered signals
all_signals = get_all_signals()
print(f"Available signals: {all_signals}")

# Get detailed signal info (mapping signal names to their classes)
signal_info = get_signal_info()
for signal_name, signal_class in signal_info.items():
    print(f"{signal_name}: {signal_class}")

Available signals: []


In [33]:
# Get raw data for an asset
btc_raw_data = analyzer.get_asset_raw_data('bitcoin')
print(f"Bitcoin raw data shape: {btc_raw_data.shape}")
print(f"Bitcoin raw data columns: {btc_raw_data.columns.tolist()}")

# Get processed data with indicators and trend classifications
btc_processed_data = analyzer.get_asset_processed_data('bitcoin')
print(f"Bitcoin processed data shape: {btc_processed_data.shape}")
print(f"Bitcoin processed data columns: {btc_processed_data.columns.tolist()}")

# Show the first few rows of processed data
btc_processed_data.head()

Bitcoin raw data shape: (4359, 8)
Bitcoin raw data columns: ['date', 'open', 'high', 'low', 'close', 'market_cap', 'total_volume', 'asset_id']
Bitcoin processed data shape: (4359, 44)
Bitcoin processed data columns: ['date', 'open', 'high', 'low', 'close', 'market_cap', 'total_volume', 'asset_id', 'MA3_close', 'RoC3_close', 'MA5_close', 'RoC5_close', 'MA7_close', 'RoC7_close', 'MA14_close', 'RoC14_close', 'MA21_close', 'RoC21_close', 'MA30_close', 'RoC30_close', 'MA45_close', 'RoC45_close', 'MA63_close', 'RoC63_close', 'MA84_close', 'RoC84_close', 'MA100_close', 'RoC100_close', 'MA120_close', 'RoC120_close', 'MA150_close', 'RoC150_close', 'MA200_close', 'RoC200_close', 'MA252_close', 'RoC252_close', 'MA365_close', 'RoC365_close', 'short_term_trend_USD', 'medium_term_trend_USD', 'long_term_trend_USD', 'overall_trend_USD', 'RSI_Signal_28_USD', 'returns_usd']


,date,open,high,low,close,market_cap,total_volume,asset_id,MA3_close,RoC3_close,...,MA252_close,RoC252_close,MA365_close,RoC365_close,short_term_trend_USD,medium_term_trend_USD,long_term_trend_USD,overall_trend_USD,RSI_Signal_28_USD,returns_usd
0,2013-04-28,135.30,135.30,135.30,135.30,1.575032e+09,0.0,bitcoin,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2013-04-29,141.96,141.96,141.96,141.96,1.501657e+09,0.0,bitcoin,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.049224
2,2013-04-30,135.30,135.30,135.30,135.30,1.298952e+09,0.0,bitcoin,137.520000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.046915
3,2013-05-01,117.00,117.00,117.00,117.00,1.148668e+09,0.0,bitcoin,131.420000,-13.525499,...,NaN,NaN,NaN,NaN,-2.0,NaN,NaN,NaN,NaN,-0.135255
4,2013-05-02,103.43,103.43,103.43,103.43,1.011066e+09,0.0,bitcoin,118.576667,-27.141448,...,NaN,NaN,NaN,NaN,-2.0,NaN,NaN,NaN,NaN,-0.115983


In [50]:
# Import specific signal classes
from soros_system.signals.trend_signals import ShortTermTrendSignal, MediumTermTrendSignal, LongTermTrendSignal
from soros_system.signals.rsi_signals import RSISignal, RSIWithRoCSignal
from soros_system.signals.volatility_signals import MarkovVolatilitySignal
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Get raw and processed data for bitcoin
btc_data = analyzer.get_asset_raw_data('bitcoin')

# Create signal instances
short_term_signal = ShortTermTrendSignal()
medium_term_signal = MediumTermTrendSignal()
long_term_signal = LongTermTrendSignal()
rsi_signal = RSISignal()
volatility_signal = MarkovVolatilitySignal()
roc_rsi_signal = RSIWithRoCSignal()

# Calculate signals
short_term_values = short_term_signal.calculate(btc_data, 'bitcoin')
medium_term_values = medium_term_signal.calculate(btc_data, 'bitcoin')
long_term_values = long_term_signal.calculate(btc_data, 'bitcoin')
rsi_values = rsi_signal.calculate(btc_data, 'bitcoin')
roc_rsi_values = roc_rsi_signal.calculate(btc_data, 'bitcoin')
volatility_values = volatility_signal.calculate(btc_data, 'bitcoin')

# Create DataFrame with all signals
signals_df = pd.DataFrame({
    'Date': btc_data['date'],
    'Price': btc_data['close'],
    'ShortTerm': short_term_values,
    'MediumTerm': medium_term_values,
    'LongTerm': long_term_values,
    'RSI': rsi_values,
    'RSI+RoC': roc_rsi_values,
    'Volatility': volatility_values
})

# Display a table with the signals
print("Signal Summary Table:")
# Convert signal values to more readable format
signals_table = signals_df.copy()
signals_table['ShortTerm'] = signals_table['ShortTerm'].map({1: 'Bullish', -1: 'Bearish', 0: 'Neutral', None: 'N/A'})
signals_table['MediumTerm'] = signals_table['MediumTerm'].map({1: 'Bullish', -1: 'Bearish', 0: 'Neutral', None: 'N/A'})
signals_table['LongTerm'] = signals_table['LongTerm'].map({1: 'Bullish', -1: 'Bearish', 0: 'Neutral', None: 'N/A'})
signals_table['RSI'] = signals_table['RSI'].map({1: 'Bullish', -1: 'Bearish', 0: 'Neutral', None: 'N/A'})
signals_table['RSI+RoC'] = signals_table['RSI+RoC'].map({1: 'Bullish', -1: 'Bearish', 0: 'Neutral', None: 'N/A'})
signals_table['Volatility'] = signals_table['Volatility'].map({1: 'Low Vol', -1: 'High Vol', 0: 'Neutral', None: 'N/A'})

# Display the last 10 rows of the signals table
display(signals_table.tail(10))

# Create separate charts for each signal using plotly
# Price chart
fig_price = go.Figure()
fig_price.add_trace(go.Scatter(
    x=signals_df['Date'],
    y=signals_df['Price'],
    mode='lines',
    name='Bitcoin Price'
))
fig_price.update_layout(
    title='Bitcoin Price',
    xaxis_title='Date',
    yaxis_title='Price (USD)',
    height=500,
    template='plotly_white'
)
fig_price.show()

# Short Term Signal
fig_short = go.Figure()
fig_short.add_trace(go.Scatter(
    x=signals_df['Date'][signals_df['ShortTerm'] == 1],
    y=[1] * sum(signals_df['ShortTerm'] == 1),
    mode='markers',
    name='Bullish',
    marker=dict(color='green', size=8)
))
fig_short.add_trace(go.Scatter(
    x=signals_df['Date'][signals_df['ShortTerm'] == -1],
    y=[-1] * sum(signals_df['ShortTerm'] == -1),
    mode='markers',
    name='Bearish',
    marker=dict(color='red', size=8)
))
fig_short.update_layout(
    title='Short Term Trend Signal',
    xaxis_title='Date',
    yaxis=dict(
        tickvals=[-1, 0, 1],
        ticktext=['Bearish', 'Neutral', 'Bullish']
    ),
    height=400,
    template='plotly_white'
)
fig_short.show()

# Medium Term Signal
fig_medium = go.Figure()
fig_medium.add_trace(go.Scatter(
    x=signals_df['Date'][signals_df['MediumTerm'] == 1],
    y=[1] * sum(signals_df['MediumTerm'] == 1),
    mode='markers',
    name='Bullish',
    marker=dict(color='green', size=8)
))
fig_medium.add_trace(go.Scatter(
    x=signals_df['Date'][signals_df['MediumTerm'] == -1],
    y=[-1] * sum(signals_df['MediumTerm'] == -1),
    mode='markers',
    name='Bearish',
    marker=dict(color='red', size=8)
))
fig_medium.update_layout(
    title='Medium Term Trend Signal',
    xaxis_title='Date',
    yaxis=dict(
        tickvals=[-1, 0, 1],
        ticktext=['Bearish', 'Neutral', 'Bullish']
    ),
    height=400,
    template='plotly_white'
)
fig_medium.show()

# Long Term Signal
fig_long = go.Figure()
fig_long.add_trace(go.Scatter(
    x=signals_df['Date'][signals_df['LongTerm'] == 1],
    y=[1] * sum(signals_df['LongTerm'] == 1),
    mode='markers',
    name='Bullish',
    marker=dict(color='green', size=8)
))
fig_long.add_trace(go.Scatter(
    x=signals_df['Date'][signals_df['LongTerm'] == -1],
    y=[-1] * sum(signals_df['LongTerm'] == -1),
    mode='markers',
    name='Bearish',
    marker=dict(color='red', size=8)
))
fig_long.update_layout(
    title='Long Term Trend Signal',
    xaxis_title='Date',
    yaxis=dict(
        tickvals=[-1, 0, 1],
        ticktext=['Bearish', 'Neutral', 'Bullish']
    ),
    height=400,
    template='plotly_white'
)
fig_long.show()

# RSI Signal
fig_rsi = go.Figure()
fig_rsi.add_trace(go.Scatter(
    x=signals_df['Date'][signals_df['RSI'] == 1],
    y=[1] * sum(signals_df['RSI'] == 1),
    mode='markers',
    name='Bullish',
    marker=dict(color='green', size=8)
))
fig_rsi.add_trace(go.Scatter(
    x=signals_df['Date'][signals_df['RSI'] == -1],
    y=[-1] * sum(signals_df['RSI'] == -1),
    mode='markers',
    name='Bearish',
    marker=dict(color='red', size=8)
))
fig_rsi.update_layout(
    title='RSI Signal',
    xaxis_title='Date',
    yaxis=dict(
        tickvals=[-1, 0, 1],
        ticktext=['Bearish', 'Neutral', 'Bullish']
    ),
    height=400,
    template='plotly_white'
)
fig_rsi.show()

fig_rsi_roc = go.Figure()
fig_rsi_roc.add_trace(go.Scatter(
    x=signals_df['Date'][signals_df['RSI+RoC'] == 1],
    y=[1] * sum(signals_df['RSI+RoC'] == 1),
    mode='markers',
    name='Bullish',
    marker=dict(color='green', size=8)
))
fig_rsi_roc.add_trace(go.Scatter(
    x=signals_df['Date'][signals_df['RSI+RoC'] == -1],
    y=[-1] * sum(signals_df['RSI+RoC'] == -1),
    mode='markers',
    name='Bearish',
    marker=dict(color='red', size=8)
))
fig_rsi_roc.update_layout(
    title='RSI+RoC Signal',
    xaxis_title='Date',
    yaxis=dict(
        tickvals=[-1, 0, 1],
        ticktext=['Bearish', 'Neutral', 'Bullish']
    ),
    height=400,
    template='plotly_white'
)
fig_rsi_roc.show()


# Volatility Signal
fig_vol = go.Figure()
fig_vol.add_trace(go.Scatter(
    x=signals_df['Date'][signals_df['Volatility'] == 1],
    y=[1] * sum(signals_df['Volatility'] == 1),
    mode='markers',
    name='Low Volatility',
    marker=dict(color='green', size=8)
))
fig_vol.add_trace(go.Scatter(
    x=signals_df['Date'][signals_df['Volatility'] == -1],
    y=[-1] * sum(signals_df['Volatility'] == -1),
    mode='markers',
    name='High Volatility',
    marker=dict(color='red', size=8)
))
fig_vol.update_layout(
    title='Volatility Signal',
    xaxis_title='Date',
    yaxis=dict(
        tickvals=[-1, 0, 1],
        ticktext=['High Vol', 'Neutral', 'Low Vol']
    ),
    height=400,
    template='plotly_white'
)
fig_vol.show()

Creating trend data for bitcoin:   0%|          | 0/2 [00:00<?, ?it/s]

Creating trend data for bitcoin:   0%|          | 0/2 [00:00<?, ?it/s]

Creating trend data for bitcoin:   0%|          | 0/2 [00:00<?, ?it/s]

Signal Summary Table:


,Date,Price,ShortTerm,MediumTerm,LongTerm,RSI,RSI+RoC,Volatility
4349,2025-03-25,87521.0,Bullish,Bearish,Bearish,Bearish,Bearish,Low Vol
4350,2025-03-26,86961.0,Bullish,Bearish,Bearish,Bullish,Bullish,Low Vol
4351,2025-03-27,87227.0,Bullish,Bearish,Bearish,Bullish,Bullish,Low Vol
4352,2025-03-28,84359.0,Bearish,Bearish,Bearish,Bullish,Bullish,Low Vol
4353,2025-03-29,82679.0,Bearish,Bearish,Bearish,Bearish,Bearish,Low Vol
4354,2025-03-30,82356.0,Bearish,Bearish,Bearish,Bearish,Bearish,Low Vol
4355,2025-03-31,82514.0,Bearish,Bearish,Bearish,Bearish,Bearish,Low Vol
4356,2025-04-01,85238.0,Bearish,Bearish,Bearish,Bearish,Bearish,Low Vol
4357,2025-04-02,82526.0,Bearish,Bearish,Bearish,Bearish,Bearish,Low Vol
4358,2025-04-03,83164.0,Bearish,Bearish,Bearish,Bearish,Bearish,Low Vol


In [53]:
# Initialize the RSI with RoC signal
roc_rsi_signal = RSIWithRoCSignal()

# Get the RSI calculator from the signal
rsi_calculator = roc_rsi_signal.rsi_calculator

# Calculate the raw RSI and RoC values before they're converted to signals
price_col = 'close'  # Using USD quote type by default
result_df = rsi_calculator.calculate_smooth_rsi(
    btc_data, price_col, 
    rsi_length=roc_rsi_signal.rsi_length, 
    roc_length=roc_rsi_signal.roc_length
)

# Create a DataFrame with dates and the RSI/RoC values
display_df = pd.DataFrame({
    'Date': btc_data['date'],
    f'RSI_{price_col}': result_df[f'RSI_{price_col}'],
    f'RoC_{price_col}': result_df[f'RoC_{price_col}']
})

# Display the raw RSI and RoC values with dates
print(f"RSI and RoC values for Bitcoin:")
print(display_df.tail(10))

# Now calculate the actual signal
roc_rsi_values = roc_rsi_signal.calculate(btc_data, 'bitcoin')

RSI and RoC values for Bitcoin:
           Date  RSI_close  RoC_close
4349 2025-03-25  48.083848  -2.505153
4350 2025-03-26  50.968755   1.365029
4351 2025-03-27  52.012738   2.720938
4352 2025-03-28  52.549889   3.164102
4353 2025-03-29  48.254895  -2.146030
4354 2025-03-30  41.990308  -8.179797
4355 2025-03-31  42.040935  -8.015829
4356 2025-04-01  47.529144  -2.382567
4357 2025-04-02  45.087343  -4.760270
4358 2025-04-03  40.962967  -8.459652
